# IEEE 1584-2018 Incident Energy Debugging

**Bug:** Incident energy equation produces ~10²⁸ J/cm² instead of ~10-50 cal/cm²

**Location:** `arc_flash_studio/calculations/ieee_1584_2018/incident_energy.py`

**Reference:** IEEE 1584-2018, Section 4.6, Equations (3)-(6)

In [1]:
import math
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Auto-reload when you edit source files
%load_ext autoreload
%autoreload 2

print(f"Project root: {project_root}")

Project root: /home/sramharack/arc_flash_studio


In [2]:
# Import from your ieee_1584_2018 module
from arc_flash_studio.calculations.ieee_1584_2018 import (
    ElectrodeConfig,
    calculate_arc_flash,
    calculate_arcing_current,
    get_energy_coefficients,
)

print("Imports successful!")

Imports successful!


## 1. Test Case: IEEE 1584-2018 Annex D.1 (4160V MV Example)

Fill in expected values from your copy of the standard.

In [7]:
# INPUTS from Annex D.1
Ibf = 15.0       # kA
Voc = 4.16       # kV
G = 104.0        # mm
D = 914.4        # mm (36")
T = 197.0        # ms
config = ElectrodeConfig.VCB

# EXPECTED VALUES - Taken from Annex D.1
expected = {
    "Iarc_600": 11.117,     # kA (Taken from Eq. D.9)
    "Iarc_2700": 12.816,    # kA (Taken from Eq. D.11)  
    "Iarc_14300": 14.116,   # kA (Taken from Eq. D.13)  
    "Iarc_final": 12.979,   # kA (Taken from Eq. D.17)
    "E_600": 8.652,         # J/cm² (Taken from Eq. D.24)
    "E_2700": 11.977,       # J/cm² (Taken from Eq. D.26)
    "E_14300": 13.367,      # J/cm² (Taken from Eq. D.28)
    "E_final": 12.152,      # J/cm² (Taken from Eq. D.32)
}

print("Expected values from IEEE 1584-2018 Annex D.1",expected)

Expected values from IEEE 1584-2018 Annex D.1 {'Iarc_600': 11.117, 'Iarc_2700': 12.816, 'Iarc_14300': 14.116, 'Iarc_final': 12.979, 'E_600': 8.652, 'E_2700': 11.977, 'E_14300': 13.367, 'E_final': 12.152}


## 2. Get Coefficients

In [8]:
k = get_energy_coefficients(config, 2700)

print("TABLE 4 COEFFICIENTS (2700V, VCB)")
print("=" * 40)
for name, value in k.items():
    print(f"{name}: {value}")

TABLE 4 COEFFICIENTS (2700V, VCB)
k1: 2.40021
k2: 0.165
k3: 0.354202
k4: -1.557e-12
k5: 4.556e-10
k6: -4.186e-08
k7: 8.346e-07
k8: 5.482e-05
k9: -0.003191
k10: 0.9729
k11: 0
k12: -1.569
k13: 0.9778


# 3. Validate the constants used in Eq. 1 from Secion 4.4

The equation to calculate the arcing current at 600V uses constants from Table 1 - Coefficients for Equation (1).

## 3. Step Through the Exponent - Find the Bug

The current implementation has:
```python
f_ibf = k4*Ibf^7 + k5*Ibf^6 + k6*Ibf^5 + k7*Ibf^4 + k8*Ibf^3 + k9*Ibf^2 + k10*Ibf
```

With k10 ≈ 0.97 and Ibf = 15, this gives k10*Ibf ≈ 14.6 added to the exponent!

In [9]:
# Approximate values for intermediate currents
Iarc_ref = 12.0    # kA (approximate)
Iarc_final = 12.0  # kA
CF = 1.2           # Typical

print("CURRENT IMPLEMENTATION - Exponent Breakdown")
print("=" * 70)

exp = k['k1']
print(f"k1                         = {exp:.6f}")

exp += k['k2'] * math.log10(G)
print(f"+ k2*lg(G)                 = {exp:.6f}")

exp += k['k3'] * math.log10(Iarc_ref)
print(f"+ k3*lg(Iarc_ref)          = {exp:.6f}")

# THE POLYNOMIAL - Current (buggy?) implementation
print("\n--- POLYNOMIAL (current implementation) ---")
f_ibf = (
    k['k4'] * Ibf**7 +
    k['k5'] * Ibf**6 +
    k['k6'] * Ibf**5 +
    k['k7'] * Ibf**4 +
    k['k8'] * Ibf**3 +
    k['k9'] * Ibf**2 +
    k['k10'] * Ibf     # <-- k10 * Ibf = 0.97 * 15 = 14.6!
)
print(f"k4*Ibf^7 = {k['k4'] * Ibf**7:.6f}")
print(f"k5*Ibf^6 = {k['k5'] * Ibf**6:.6f}")
print(f"k6*Ibf^5 = {k['k6'] * Ibf**5:.6f}")
print(f"k7*Ibf^4 = {k['k7'] * Ibf**4:.6f}")
print(f"k8*Ibf^3 = {k['k8'] * Ibf**3:.6f}")
print(f"k9*Ibf^2 = {k['k9'] * Ibf**2:.6f}")
print(f"k10*Ibf  = {k['k10'] * Ibf:.6f}  <<< PROBLEM?")
print(f"\nTotal polynomial f(Ibf) = {f_ibf:.6f}")

exp += f_ibf
print(f"\n+ f(Ibf)                   = {exp:.6f}")

exp += k['k11'] * math.log10(Ibf)
print(f"+ k11*lg(Ibf)              = {exp:.6f}")

exp += k['k12'] * math.log10(D)
print(f"+ k12*lg(D)                = {exp:.6f}")

exp += k['k13'] * math.log10(Iarc_final)
print(f"+ k13*lg(Iarc_final)       = {exp:.6f}")

exp += math.log10(1/CF)
print(f"+ lg(1/CF)                 = {exp:.6f}")

print("\n" + "=" * 70)
print(f"FINAL EXPONENT = {exp:.4f}")
print(f"10^exponent = {10**exp:.4e}")

E = (12.552/50) * T * (10**exp)
print(f"\nE = {E:.4e} J/cm²")
print(f"E = {E/4.184:.4e} cal/cm²")

CURRENT IMPLEMENTATION - Exponent Breakdown
k1                         = 2.400210
+ k2*lg(G)                 = 2.733021
+ k3*lg(Iarc_ref)          = 3.115269

--- POLYNOMIAL (current implementation) ---
k4*Ibf^7 = -0.000266
k5*Ibf^6 = 0.005190
k6*Ibf^5 = -0.031787
k7*Ibf^4 = 0.042252
k8*Ibf^3 = 0.185018
k9*Ibf^2 = -0.717975
k10*Ibf  = 14.593500  <<< PROBLEM?

Total polynomial f(Ibf) = 14.075930

+ f(Ibf)                   = 17.191199
+ k11*lg(Ibf)              = 17.191199
+ k12*lg(D)                = 12.545176
+ k13*lg(Iarc_final)       = 13.600400
+ lg(1/CF)                 = 13.521218

FINAL EXPONENT = 13.5212
10^exponent = 3.3206e+13

E = 1.6422e+15 J/cm²
E = 3.9250e+14 cal/cm²


## 4. Try Alternative: k10 as Constant (not × Ibf)

What if the polynomial should be:
```
f(Ibf) = k4*Ibf^6 + k5*Ibf^5 + k6*Ibf^4 + k7*Ibf^3 + k8*Ibf^2 + k9*Ibf + k10
```

In [ ]:
print("ALTERNATIVE: k10 as constant term")
print("=" * 70)

# Reset exponent
exp_alt = (
    k['k1'] +
    k['k2'] * math.log10(G) +
    k['k3'] * math.log10(Iarc_ref)
)

# Alternative polynomial: k10 is constant, not multiplied by Ibf
f_ibf_alt = (
    k['k4'] * Ibf**6 +   # Changed from Ibf^7
    k['k5'] * Ibf**5 +   # Changed from Ibf^6
    k['k6'] * Ibf**4 +   # etc.
    k['k7'] * Ibf**3 +
    k['k8'] * Ibf**2 +
    k['k9'] * Ibf +
    k['k10']             # Constant term!
)

print(f"Alternative polynomial f(Ibf) = {f_ibf_alt:.6f}")

exp_alt += f_ibf_alt
exp_alt += k['k11'] * math.log10(Ibf)
exp_alt += k['k12'] * math.log10(D)
exp_alt += k['k13'] * math.log10(Iarc_final)
exp_alt += math.log10(1/CF)

print(f"Final exponent = {exp_alt:.4f}")

E_alt = (12.552/50) * T * (10**exp_alt)
print(f"\nE = {E_alt:.4f} J/cm²")
print(f"E = {E_alt/4.184:.4f} cal/cm²")
print(f"\nExpected E_2700 from Annex D.1: {expected['E_2700']} J/cm²")

## 5. Verify Against IEEE 1584-2018

Open Section 4.6 and check:

1. **What are the exact polynomial powers?**
   - k4 × Ibf^?
   - k5 × Ibf^?
   - ...
   - k10 × Ibf^? (or just k10?)

2. **Write the correct equation here:**

In [10]:
# After checking IEEE 1584-2018, implement the CORRECT equation:

def calculate_energy_CORRECT(k, Ibf, G, D, T, Iarc_ref, Iarc_final, CF):
    """
    CORRECT implementation after verifying against standard.
    
    TODO: Update this based on what you find in IEEE 1584-2018 Section 4.6
    """
    exponent = (
        k['k1'] +
        k['k2'] * math.log10(G) +
        k['k3'] * math.log10(Iarc_ref)
    )
    
    # TODO: Fix the polynomial based on the standard
    # Current (buggy):
    # f_ibf = k4*Ibf^7 + k5*Ibf^6 + ... + k10*Ibf
    #
    # Likely correct:
    # f_ibf = k4*Ibf^6 + k5*Ibf^5 + ... + k9*Ibf + k10
    
    f_ibf = (
        k['k4'] * Ibf**6 +
        k['k5'] * Ibf**5 +
        k['k6'] * Ibf**4 +
        k['k7'] * Ibf**3 +
        k['k8'] * Ibf**2 +
        k['k9'] * Ibf +
        k['k10']  # Constant
    )
    exponent += f_ibf
    
    exponent += k['k11'] * math.log10(Ibf)
    exponent += k['k12'] * math.log10(D)
    exponent += k['k13'] * math.log10(Iarc_final)
    exponent += math.log10(1/CF)
    
    energy = (12.552/50) * T * (10**exponent)
    return energy

# Test it
E_correct = calculate_energy_CORRECT(k, Ibf, G, D, T, Iarc_ref, Iarc_final, CF)
print(f"Corrected E = {E_correct:.4f} J/cm² = {E_correct/4.184:.4f} cal/cm²")

Corrected E = 119.6445 J/cm² = 28.5957 cal/cm²


## 6. Once Fixed: Full Validation

After updating `incident_energy.py`, run this to validate:

In [11]:
#After fixing the source code, test the full calculation
result = calculate_arc_flash(
    ibf_ka=Ibf,
    voc_kv=Voc,
    gap_mm=G,
    working_distance_mm=D,
    arc_duration_ms=T,
    height_mm=508.0,
    width_mm=508.0,
    depth_mm=508.0,
    config=config
)

print(f"Iarc: {result.iarc_ka:.2f} kA (expected: {expected['Iarc_final']})")
print(f"E: {result.incident_energy_j_cm2:.2f} J/cm² (expected: {expected['E_final']})")

Iarc: 12.98 kA (expected: 12.979)
E: 2064005459514905.50 J/cm² (expected: 12.152)
